# EDL Full Experiment (parallel)

10-fold cross-validation comparison of 8 LDL models across 3 datasets
(`SJAFFE`, `SBU_3DFE`, `Human_Gene`), parallelised across CPU workers or
GPUs via `loky`.

The actual training loop lives in `edl_workers.py` next to this notebook
so loky subprocesses can `import edl_workers` cleanly. TensorFlow is
imported lazily inside each worker after `CUDA_VISIBLE_DEVICES` is pinned.

For each (model, dataset) pair we record six distributional metrics
(`chebyshev`, `clark`, `canberra`, `kl_divergence`, `cosine`, `intersection`)
across 10 folds with a 10% test split per fold, then summarise as
mean ± std.

For the evidential models (`EDL_LDL`, `BEDL_LDL`) and `SNEFY_LDL` we also
report:

- **Mean uncertainty** — average per-sample uncertainty on the test set
  (lower = the model is more confident).
- **Uncertainty calibration (Spearman ρ)** — rank correlation between
  per-sample uncertainty and per-sample KL divergence error.
  Higher = uncertainty tracks error better.


In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import multiprocessing as mp
from collections import defaultdict
from concurrent.futures import as_completed

import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from loky import get_reusable_executor

from pyldl.utils import load_dataset

import edl_workers
from edl_workers import init_worker, run_one_fold, MODEL_NAMES, METRICS


## Configuration

`GPU_IDS` — list of CUDA device ids to use, one worker per id.
Set to `[]` for CPU-only; in that case `N_WORKERS` controls the pool size.

`N_EPOCHS` is passed to every model whose `fit()` accepts it (everything
except `SA_BFGS`).


In [ ]:
DATASETS = ['SJAFFE', 'SBU_3DFE', 'Human_Gene']
N_SPLITS = 10
N_EPOCHS = 100
RANDOM_STATE = 0

# --- Parallel config -----------------------------------------------------
GPU_IDS = []                                 # e.g. [0, 1, 2, 3] for 4 GPUs
N_WORKERS = len(GPU_IDS) if GPU_IDS else max(1, (os.cpu_count() or 2) // 2)
# -------------------------------------------------------------------------


## Build the worker pool

Each worker pulls one entry off `gpu_queue` exactly once at startup
(loky's `initializer`) and pins `CUDA_VISIBLE_DEVICES` before TF sees a
GPU. `reuse=False` forces a fresh pool if you re-run this cell after
changing the config.


In [ ]:
mgr = mp.Manager()
gpu_queue = mgr.Queue()

slots = list(GPU_IDS) if GPU_IDS else [None] * N_WORKERS
assert len(slots) == N_WORKERS, 'one queue slot per worker'
for g in slots:
    gpu_queue.put(g)

executor = get_reusable_executor(
    max_workers=N_WORKERS,
    initializer=init_worker,
    initargs=(gpu_queue,),
    reuse=False,
)
print(f'pool ready: {N_WORKERS} workers, gpu_ids={slots}')


## Build the job list

`KFold` splits are generated in the parent (deterministic, cheap) and the
fold slices are passed to workers as numpy arrays. Loky memmaps large
numpy arrays automatically, so this is fast even for the bigger datasets.


In [ ]:
jobs = []
for dataset_name in DATASETS:
    X, D = load_dataset(dataset_name, dir='dataset')
    kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    for fold_idx, (train_idx, test_idx) in enumerate(kf.split(X), start=1):
        Xtr, Xte = X[train_idx], X[test_idx]
        Dtr, Dte = D[train_idx], D[test_idx]
        for model_name in MODEL_NAMES:
            jobs.append((dataset_name, model_name, fold_idx, Xtr, Dtr, Xte, Dte))

total = len(jobs)
print(f'queued {total} jobs ({len(DATASETS)} datasets × {N_SPLITS} folds × {len(MODEL_NAMES)} models)')


## Submit + collect

Submission is non-blocking; results stream back via `as_completed` so the
log shows progress as folds finish. Per-fold failures are caught and
printed but don't stop the run.


In [ ]:
futures = {
    executor.submit(run_one_fold, ds, m, fi, Xtr, Dtr, Xte, Dte, N_EPOCHS): (ds, m, fi)
    for (ds, m, fi, Xtr, Dtr, Xte, Dte) in jobs
}

raw_results = []
for i, fut in enumerate(as_completed(futures), start=1):
    ds, m, fi = futures[fut]
    try:
        raw_results.append(fut.result())
        status = 'ok'
    except Exception as e:
        status = f'FAILED ({type(e).__name__}: {e})'
    print(f'[{i:4d}/{total}] {ds:12s} | fold {fi:2d} | {m:30s} {status}')


## Bucket results into per-model DataFrames

`per_model_results[(dataset, model_name)]` is a DataFrame with one row per
fold; columns are the recorded metrics.


In [ ]:
buckets = defaultdict(list)
for r in raw_results:
    buckets[(r['dataset'], r['model'])].append(r['scores'])

per_model_results = {key: pd.DataFrame(rows) for key, rows in buckets.items()}


## Per-model fold tables

Inspect any single (dataset, model) DataFrame:


In [ ]:
per_model_results[('SJAFFE', 'EDL_LDL (loglikelihood)')]


## Combined summary — mean ± std across folds

One row per (dataset, model); columns are `metric_mean` / `metric_std`.


In [ ]:
def summarize(df):
    out = {}
    for col in df.columns:
        out[f'{col}_mean'] = df[col].mean()
        out[f'{col}_std']  = df[col].std()
    return out


summary_rows = []
for (dataset_name, model_name), df in per_model_results.items():
    if df.empty:
        continue
    row = {'dataset': dataset_name, 'model': model_name, **summarize(df)}
    summary_rows.append(row)

summary = pd.DataFrame(summary_rows).set_index(['dataset', 'model'])
summary


### Compact view: `mean ± std` per metric


In [ ]:
def fmt(mean, std):
    if pd.isna(mean):
        return ''
    return f'{mean:.4f} ± {std:.4f}'


compact_rows = []
for (dataset_name, model_name), df in per_model_results.items():
    if df.empty:
        continue
    row = {'dataset': dataset_name, 'model': model_name}
    for col in df.columns:
        row[col] = fmt(df[col].mean(), df[col].std())
    compact_rows.append(row)

compact = pd.DataFrame(compact_rows).set_index(['dataset', 'model'])
compact


## Uncertainty results (EDL_LDL, BEDL_LDL, SNEFY_LDL only)

- `mean_uncertainty` — average per-sample uncertainty on test (model-specific
  scale; lower = more confident).
- `uncertainty_calibration` — Spearman ρ between per-sample uncertainty and
  per-sample KL divergence error. Higher = uncertainty better predicts error.


In [ ]:
uncertainty_models = {
    'EDL_LDL (loglikelihood)', 'EDL_LDL (bayes_mse)',
    'BEDL_LDL (loglikelihood)', 'BEDL_LDL (bayes_mse)',
    'SNEFY_LDL',
}

uncertainty_rows = []
for (dataset_name, model_name), df in per_model_results.items():
    if model_name not in uncertainty_models or df.empty:
        continue
    if 'mean_uncertainty' not in df.columns:
        continue
    uncertainty_rows.append({
        'dataset': dataset_name,
        'model': model_name,
        'mean_uncertainty':        fmt(df['mean_uncertainty'].mean(),        df['mean_uncertainty'].std()),
        'uncertainty_calibration': fmt(df['uncertainty_calibration'].mean(), df['uncertainty_calibration'].std()),
    })

uncertainty_summary = pd.DataFrame(uncertainty_rows).set_index(['dataset', 'model'])
uncertainty_summary


## Shut down the pool

Loky reuses pools by default; close it explicitly when you're done so the
worker processes (and any GPU memory they hold) are released.


In [ ]:
executor.shutdown(wait=True, kill_workers=True)
